In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------------
# 1) โหลดและทำความสะอาดข้อมูล
# ---------------------------------------------------------------
FILE = "Salary_Data_7.csv"   # แก้ path ตามที่เก็บไฟล์
df = pd.read_csv(FILE)

# ตั้งชื่อคอลัมน์ให้ตรงกับในรูป (salary_eur, age, exp_total_years ...)
df = df.rename(columns={
    "Salary": "salary",
    "Age": "age",
    "Years of Experience": "exp_total_years",
    "Education Level": "education",   # ใช้แทน "ระดับตำแหน่ง" (ไม่มีคอลัมน์ seniority ในไฟล์นี้)
    "Gender": "gender",
    "Job Title": "job_title",         # ใช้แทน "เมืองที่ทำงาน" (ไม่มีคอลัมน์ city ในไฟล์นี้)
})

# ลบแถวที่ข้อมูลหาย
df = df.dropna(subset=["salary", "age", "exp_total_years", "education", "gender", "job_title"])

# รวมชื่อระดับการศึกษาที่พิมพ์ไม่เหมือนกันให้เป็นค่าเดียว
df["education"] = df["education"].replace({
    "Bachelor's": "Bachelor's Degree",
    "Master's": "Master's Degree",
    "phD": "PhD",
})

# เรียงระดับการศึกษาจากต่ำไปสูง
edu_order = ["High School", "Bachelor's Degree", "Master's Degree", "PhD"]
df["education"] = pd.Categorical(df["education"], categories=edu_order, ordered=True)

sns.set_theme(style="whitegrid")

# ---------------------------------------------------------------
# 2) ฮิสโตแกรม: การแจกแจงของ salary และ age
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(df["salary"], bins=40, kde=True, color="#4C72B0", ax=axes[0])
axes[0].set_title("Histogram: Salary")
axes[0].set_xlabel("Salary")
axes[0].set_ylabel("Count")

sns.histplot(df["age"], bins=30, kde=True, color="#55A868", ax=axes[1])
axes[1].set_title("Histogram: Age")
axes[1].set_xlabel("Age")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig("1_histogram.png", dpi=150)

# ---------------------------------------------------------------
# 3) แผนภาพกล่อง: salary เทียบตามระดับการศึกษา และเพศ
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="education", y="salary", hue="education",
            palette="Set2", legend=False, ax=axes[0])
axes[0].set_title("Boxplot: Salary by Education Level")
axes[0].set_xlabel("Education Level")
axes[0].set_ylabel("Salary")

sns.boxplot(data=df, x="gender", y="salary", hue="gender",
            palette="Set1", legend=False, ax=axes[1])
axes[1].set_title("Boxplot: Salary by Gender")
axes[1].set_xlabel("Gender")
axes[1].set_ylabel("Salary")

plt.tight_layout()
plt.savefig("2_boxplot.png", dpi=150)

# ---------------------------------------------------------------
# 4) กราฟแท่ง: เงินเดือนเฉลี่ยแยกตามระดับการศึกษา และตำแหน่งงาน
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 4.1 เฉลี่ยตามระดับการศึกษา
mean_edu = df.groupby("education", observed=True)["salary"].mean().reset_index()
sns.barplot(data=mean_edu, x="education", y="salary", hue="education",
            palette="Blues_d", legend=False, ax=axes[0])
axes[0].set_title("Mean Salary by Education Level")
axes[0].set_xlabel("Education Level")
axes[0].set_ylabel("Mean Salary")
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():,.0f}",
                     (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha="center", va="bottom", fontsize=9)

# 4.2 เฉลี่ยตามตำแหน่งงาน (Top 10 ตำแหน่งที่มีข้อมูลมากสุด)
top_jobs = df["job_title"].value_counts().head(10).index
mean_job = (df[df["job_title"].isin(top_jobs)]
            .groupby("job_title")["salary"].mean()
            .sort_values(ascending=False).reset_index())
sns.barplot(data=mean_job, y="job_title", x="salary", hue="job_title",
            palette="viridis", legend=False, ax=axes[1])
axes[1].set_title("Mean Salary by Job Title (Top 10 most common)")
axes[1].set_xlabel("Mean Salary")
axes[1].set_ylabel("Job Title")

plt.tight_layout()
plt.savefig("3_barchart.png", dpi=150)

# ---------------------------------------------------------------
# 5) กราฟการกระจาย (Scatter): ประสบการณ์ vs salary
# ---------------------------------------------------------------
plt.figure(figsize=(8, 5.5))
sns.scatterplot(data=df, x="exp_total_years", y="salary",
                hue="education", alpha=0.5, s=25)
sns.regplot(data=df, x="exp_total_years", y="salary", scatter=False,
            color="red", line_kws={"linewidth": 2, "label": "Trend line"})
plt.title("Scatter: Years of Experience vs Salary")
plt.xlabel("Years of Experience")
plt.ylabel("Salary")
plt.legend(title="Education", fontsize=8)
plt.tight_layout()
plt.savefig("4_scatter.png", dpi=150)

# ค่าสหสัมพันธ์ระหว่างประสบการณ์กับเงินเดือน
corr = df["exp_total_years"].corr(df["salary"])
print(f"Correlation (experience vs salary): {corr:.3f}")

plt.show()